In [1]:
%%capture
!pip install dlt[duckdb]
!pip install jupysql
!pip install duckdb
!pip install duckdb-engine

In [2]:
%load_ext sql

In [3]:
# Import libraries
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

import dlt
import duckdb
import pandas as pd

In [4]:
# Explore header type
import requests
response = requests.get("https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api")
response.headers

{'content-type': 'application/json', 'function-execution-id': 'oyekpag5o48e', 'x-cloud-trace-context': 'd837fc4ebb508aefe7ff27b170e88ff0', 'date': 'Sat, 28 Feb 2026 16:39:23 GMT', 'server': 'Google Frontend', 'Content-Length': '396225'}

In [5]:
# explore pagination used
from dlt.sources.helpers.rest_client import RESTClient


client = RESTClient(
    base_url="https://us-central1-dlthub-analytics.cloudfunctions.net",
)

for page in client.paginate("/data_engineering_zoomcamp_api"):
    print(f"Paginator used: {page.paginator}")

2026-02-28 16:39:28,523|[WARNING]|22768|138081838825472|dlt|client.py|detect_paginator:365|Fallback `paginator` used: `SinglePagePaginator at 7d9577f2a210`. Please provide paginator manually.


Paginator used: SinglePagePaginator at 7d9577f2a210


In [6]:
import dlt
import duckdb
from dlt.sources.rest_api import rest_api_source

@dlt.source
def taxi_source():
    config = {
        "client": {
            "base_url": "https://us-central1-dlthub-analytics.cloudfunctions.net",
            "headers": {
                 "Content-Type": "application/json"
            }
        },
        "resources": [
            {
                "name": "taxi",
                "endpoint": {
                    "path": "data_engineering_zoomcamp_api",
                    "paginator": {
                        "type": "page_number",
                        "base_page": 1,
                        "total_path": None # Stops when an empty page is returned
                    }
                }
            }
        ]
    }

    return rest_api_source(config)

pipeline = dlt.pipeline(
    pipeline_name="taxi_pipeline",
    destination="duckdb",
    dataset_name="taxi_data",
)

load_info = pipeline.run(taxi_source())
print(load_info)

2026-02-28 16:39:54,288|[WARNING]|22768|138081838825472|dlt|validate.py|verify_normalized_table:91|In schema `rest_api`: The following columns in table 'taxi' did not receive any data during this load and therefore could not have their types inferred:
  - rate_code
  - mta_tax

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'rate_code': {'data_type': 'text'}})



Pipeline taxi_pipeline load step completed in 2.55 seconds
1 load package(s) were loaded to destination duckdb and into dataset taxi_data
The duckdb destination used duckdb:////content/taxi_pipeline.duckdb location to store data
Load package 1772296768.899918 is LOADED and contains no failed jobs


**Access table using dlt**

In [7]:
dataset = pipeline.dataset().tables

In [8]:
dataset

['taxi', '_dlt_version', '_dlt_loads', '_dlt_pipeline_state']

In [9]:

taxi_df = pipeline.dataset().table("taxi").df()

In [10]:
taxi_df.head()

,end_lat,end_lon,fare_amt,passenger_count,payment_type,start_lat,start_lon,tip_amt,tolls_amt,total_amt,trip_distance,trip_dropoff_date_time,trip_pickup_date_time,surcharge,vendor_name,_dlt_load_id,_dlt_id,store_and_forward
0,40.742963,-73.980072,45.0,1,Credit,40.641525,-73.787442,9.0,4.15,58.15,17.52,2009-06-14 23:48:00+00:00,2009-06-14 23:23:00+00:00,0.0,VTS,1772296768.899918,w2zGj6SbFFYjZQ,NaN
1,40.740187,-74.005698,6.5,1,Credit,40.722065,-74.009767,1.0,0.00,8.50,1.56,2009-06-18 17:43:00+00:00,2009-06-18 17:35:00+00:00,1.0,VTS,1772296768.899918,XNkaqL3VXOS+LA,NaN
2,40.718043,-74.004745,12.5,5,Credit,40.761945,-73.983038,2.0,0.00,15.50,3.37,2009-06-10 18:27:00+00:00,2009-06-10 18:08:00+00:00,1.0,VTS,1772296768.899918,ME8E3UWwG1R3hw,NaN
3,40.739637,-73.985233,4.9,1,CASH,40.749802,-73.992247,0.0,0.00,5.40,1.11,2009-06-14 23:58:00+00:00,2009-06-14 23:54:00+00:00,0.5,VTS,1772296768.899918,MLPBSb2ovlniTA,NaN
4,40.730032,-73.852693,25.7,1,CASH,40.776825,-73.949233,0.0,4.15,29.85,11.09,2009-06-13 13:23:00+00:00,2009-06-13 13:01:00+00:00,0.0,VTS,1772296768.899918,+7EXMSUe+gyPSQ,NaN


**Access tables using SQL**

In [11]:
%%sql
duckdb:////content/taxi_pipeline.duckdb

In [12]:
%%sql
SELECT database_name, schema_name, table_name
FROM duckdb_tables();

,database_name,schema_name,table_name
0,taxi_pipeline,taxi_data,taxi
1,taxi_pipeline,taxi_data,_dlt_loads
2,taxi_pipeline,taxi_data,_dlt_pipeline_state
3,taxi_pipeline,taxi_data,_dlt_version


In [13]:
%%sql
SELECT * FROM taxi_data.taxi LIMIT 5;

,end_lat,end_lon,fare_amt,passenger_count,payment_type,start_lat,start_lon,tip_amt,tolls_amt,total_amt,trip_distance,trip_dropoff_date_time,trip_pickup_date_time,surcharge,vendor_name,_dlt_load_id,_dlt_id,store_and_forward
0,40.742963,-73.980072,45.0,1,Credit,40.641525,-73.787442,9.0,4.15,58.15,17.52,2009-06-14 23:48:00+00:00,2009-06-14 23:23:00+00:00,0.0,VTS,1772296768.899918,w2zGj6SbFFYjZQ,NaN
1,40.740187,-74.005698,6.5,1,Credit,40.722065,-74.009767,1.0,0.00,8.50,1.56,2009-06-18 17:43:00+00:00,2009-06-18 17:35:00+00:00,1.0,VTS,1772296768.899918,XNkaqL3VXOS+LA,NaN
2,40.718043,-74.004745,12.5,5,Credit,40.761945,-73.983038,2.0,0.00,15.50,3.37,2009-06-10 18:27:00+00:00,2009-06-10 18:08:00+00:00,1.0,VTS,1772296768.899918,ME8E3UWwG1R3hw,NaN
3,40.739637,-73.985233,4.9,1,CASH,40.749802,-73.992247,0.0,0.00,5.40,1.11,2009-06-14 23:58:00+00:00,2009-06-14 23:54:00+00:00,0.5,VTS,1772296768.899918,MLPBSb2ovlniTA,NaN
4,40.730032,-73.852693,25.7,1,CASH,40.776825,-73.949233,0.0,4.15,29.85,11.09,2009-06-13 13:23:00+00:00,2009-06-13 13:01:00+00:00,0.0,VTS,1772296768.899918,+7EXMSUe+gyPSQ,NaN


In [14]:
start_date = taxi_df['trip_pickup_date_time'].min()
end_date = taxi_df['trip_pickup_date_time'].max()

print(f"Start date of the dataset: {start_date}")
print(f"End date of the dataset: {end_date}")

Start date of the dataset: 2009-06-01 11:33:00+00:00
End date of the dataset: 2009-06-30 23:58:00+00:00


In [15]:
credit_card_trips = taxi_df[taxi_df['payment_type'] == 'Credit'].shape[0]
total_trips = taxi_df.shape[0]

proportion_credit_card = credit_card_trips / total_trips

print(f"Proportion of trips paid with credit card: {proportion_credit_card:.2%}")

Proportion of trips paid with credit card: 26.66%


In [18]:
total_tip_amount = taxi_df['tip_amt'].sum()
print(f"Total amount of money generated in tips: ${total_tip_amount:.2f}")

Total amount of money generated in tips: $6063.41


In [20]:
%%sql
SELECT SUM(tip_amt) AS total_tip_amount FROM taxi_data.taxi;

,total_tip_amount
0,6063.41
